# Quantum Graph Encoding for Molecular Representation in Drug Discovery


## Environment


In [4]:
import sys, subprocess, importlib.util, time
from datetime import datetime

_t0 = time.perf_counter()
print(f"[START] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | Environment", flush=True)

needed = {
    "kagglehub": "kagglehub",
    "rdkit": "rdkit",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "torch": "torch",
    "matplotlib": "matplotlib",
}
missing = [pkg for module, pkg in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import os, json, math, random, hashlib, urllib.request, shutil, warnings, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Iterable
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, mean_squared_error, mean_absolute_error, r2_score
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
import matplotlib.pyplot as plt
import kagglehub

warnings.filterwarnings("ignore")

def stamp_start(label):
    started = time.perf_counter()
    print(f"[START] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {label}", flush=True)
    return started

def stamp_end(label, started):
    print(f"[DONE] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {label} | {time.perf_counter()-started:.2f}s", flush=True)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
stamp_end("Environment", _t0)


[START] 2026-09-20 23:12:11 | Environment
Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: True
[DONE] 2026-09-20 23:12:11 | Environment | 0.00s


## Configuration


In [5]:
_t0 = stamp_start("Configuration")

@dataclass
class Config:
    source_handle: str = "abdulbasitbaloch/qgme-v20-final-fresh-state"
    source_version: str = "v20_fresh_nested_safe_topology_quantum_18jobs"
    work_root: str = "/kaggle/working/qgme_hybrid_qgnn_vqe"
    split_seed: int = 20260805
    folds: int = 3
    seeds: tuple = (19, 53)
    batch_size: int = 48
    hidden_dim: int = 256
    graph_layers: int = 6
    attention_heads: int = 8
    dropout: float = 0.15
    bottleneck_dim: int = 64
    n_qubits: int = 6
    n_electrons: int = 3
    vqe_depth: int = 8
    vqe_iterations: int = 200
    vqe_learning_rate: float = 0.01
    shots: int = 1024
    pca_components: int = 10
    graph_epochs: int = 500
    graph_patience: int = 40
    graph_learning_rate: float = 5e-4
    fusion_epochs: int = 250
    fusion_patience: int = 30
    fusion_learning_rate: float = 8e-4
    weight_decay: float = 2e-5
    max_grad_norm: float = 2.0

CFG = Config()
ROOT = Path(CFG.work_root)
RAW = ROOT / "raw"
CACHE = ROOT / "cache"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
for p in (ROOT, RAW, CACHE, CHECKPOINTS, RESULTS, FIGURES):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_SPECS = {
    "BBBP": {
        "urls": (
            "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv",
            "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/BBBP.csv",
        ),
        "smiles_columns": ("smiles",),
        "target_columns": ("p_np",),
        "task": "classification",
    },
    "BACE": {
        "urls": (
            "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/bace.csv",
            "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/bace.csv",
        ),
        "smiles_columns": ("mol", "smiles"),
        "target_columns": ("class", "Class"),
        "task": "classification",
    },
    "ESOL": {
        "urls": (
            "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv",
            "https://deepchemdata.s3.us-west-1.amazonaws.com/datasets/delaney-processed.csv",
        ),
        "smiles_columns": ("smiles",),
        "target_columns": (
            "measured log solubility in mols per litre",
            "measured log solubility in mols per liter",
        ),
        "task": "regression",
    },
}

MODEL_MAP = {
    "Fingerprint MLP": "MLP",
    "Directed Bond Graph Network": "GNN",
    "Proposed Motif-Topology Quantum Fusion": "Proposed Topology-Aware Quantum",
}
BASE_NAME = "Hybrid QGNN–VQE"

PAIR_LIST = [(i, j) for i in range(CFG.n_qubits) for j in range(i + 1, CFG.n_qubits)]
OCCUPIED = list(range(CFG.n_electrons))
VIRTUAL = list(range(CFG.n_electrons, CFG.n_qubits))
SINGLES = [(i, a) for i in OCCUPIED for a in VIRTUAL]
DOUBLES = [(i, j, a, b) for i in OCCUPIED for j in OCCUPIED if i < j for a in VIRTUAL for b in VIRTUAL if a < b]
N_VQE_PARAMS = len(SINGLES) + len(DOUBLES)


def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(json.dumps(asdict(CFG), indent=2))
print("Device:", DEVICE)
print("Model:", BASE_NAME)
print("UCCSD parameters:", N_VQE_PARAMS)
stamp_end("Configuration", _t0)


[START] 2026-09-20 23:12:11 | Configuration
{
  "source_handle": "abdulbasitbaloch/qgme-v20-final-fresh-state",
  "source_version": "v20_fresh_nested_safe_topology_quantum_18jobs",
  "work_root": "/kaggle/working/qgme_hybrid_qgnn_vqe",
  "split_seed": 20260805,
  "folds": 3,
  "seeds": [
    19,
    53
  ],
  "batch_size": 48,
  "hidden_dim": 256,
  "graph_layers": 6,
  "attention_heads": 8,
  "dropout": 0.15,
  "bottleneck_dim": 64,
  "n_qubits": 6,
  "n_electrons": 3,
  "vqe_depth": 8,
  "vqe_iterations": 200,
  "vqe_learning_rate": 0.01,
  "shots": 1024,
  "pca_components": 10,
  "graph_epochs": 500,
  "graph_patience": 40,
  "graph_learning_rate": 0.0005,
  "fusion_epochs": 250,
  "fusion_patience": 30,
  "fusion_learning_rate": 0.0008,
  "weight_decay": 2e-05,
  "max_grad_norm": 2.0
}
Device: cuda
Model: Hybrid QGNN–VQE
UCCSD parameters: 18
[DONE] 2026-09-20 23:12:11 | Configuration | 0.00s


## Prior Results


In [6]:
_t0 = stamp_start("Prior Results")

def find_source_files():
    local_roots = [
        Path("/kaggle/working/qgme_dissertation_v20_final"),
        Path("/kaggle/working"),
    ]
    candidates = []
    for root in local_roots:
        if root.exists():
            candidates.extend(root.rglob("all_job_metrics.csv"))
    selected = [p for p in candidates if CFG.source_version in str(p)]
    if not selected:
        downloaded = Path(kagglehub.dataset_download(CFG.source_handle))
        selected = [p for p in downloaded.rglob("all_job_metrics.csv") if CFG.source_version in str(p)]
    if not selected:
        raise FileNotFoundError("Previous dissertation metrics were not found.")
    metrics_path = sorted(selected, key=lambda p: len(str(p)))[0]
    predictions_candidates = list(metrics_path.parent.rglob("all_predictions.csv"))
    if predictions_candidates:
        predictions_path = predictions_candidates[0]
    else:
        predictions_path = metrics_path.parent / "all_predictions.csv"
    return metrics_path, predictions_path

source_metrics_path, source_predictions_path = find_source_files()
prior_metrics = pd.read_csv(source_metrics_path)
prior_metrics = prior_metrics[prior_metrics["model"].isin(MODEL_MAP)].copy()
prior_metrics["model"] = prior_metrics["model"].map(MODEL_MAP)

prior_predictions = pd.DataFrame()
if source_predictions_path.exists():
    prior_predictions = pd.read_csv(source_predictions_path)
    prior_predictions = prior_predictions[prior_predictions["model"].isin(MODEL_MAP)].copy()
    prior_predictions["model"] = prior_predictions["model"].map(MODEL_MAP)

print("Source:", source_metrics_path)
print(prior_metrics.groupby(["dataset", "model"]).size().unstack(fill_value=0))
stamp_end("Prior Results", _t0)


[START] 2026-09-20 23:12:11 | Prior Results
Source: /kaggle/input/datasets/abdulbasitbaloch/qgme-v20-final-fresh-state/v20_fresh_nested_safe_topology_quantum_18jobs/full/results/all_job_metrics.csv
model    GNN  MLP  Proposed Topology-Aware Quantum
dataset                                           
BACE       6    6                                6
BBBP       6    6                                6
ESOL       6    6                                6
[DONE] 2026-09-20 23:12:11 | Prior Results | 0.25s


## Data


In [7]:
_t0 = stamp_start("Data")

def resolve_column(columns, candidates):
    exact = {str(c): str(c) for c in columns}
    lower = {str(c).strip().lower(): str(c) for c in columns}
    for c in candidates:
        if c in exact:
            return exact[c]
        if c.strip().lower() in lower:
            return lower[c.strip().lower()]
    raise KeyError((columns, candidates))

def canonicalize_smiles(value):
    if not isinstance(value, str) or not value.strip():
        return None, None
    mol = Chem.MolFromSmiles(value)
    if mol is None:
        return None, None
    Chem.SanitizeMol(mol)
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True), mol

def molecular_group(mol, canonical):
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=True)
    if scaffold:
        return f"murcko::{scaffold}"
    generic = MurckoScaffold.MakeScaffoldGeneric(Chem.Mol(mol))
    generic_smiles = Chem.MolToSmiles(generic, canonical=True)
    if generic_smiles:
        return f"acyclic_generic::{generic_smiles}"
    return f"acyclic_exact::{hashlib.sha1(canonical.encode()).hexdigest()[:16]}"

def download_csv(name):
    destination = RAW / f"{name}.csv"
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    errors = []
    for url in DATASET_SPECS[name]["urls"]:
        try:
            tmp = destination.with_suffix(".download")
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, destination)
            return destination
        except Exception as exc:
            errors.append(str(exc))
            destination.with_suffix(".download").unlink(missing_ok=True)
    raise RuntimeError(f"{name} download failed: {' | '.join(errors)}")

def frame_fingerprint(frame):
    payload = "|".join(f"{sid}:{float(y):.12g}" for sid, y in zip(frame["sample_id"], frame["target"]))
    return hashlib.sha256(payload.encode()).hexdigest()

def load_frame(name):
    cache_path = CACHE / f"{name}_frame.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path)
    raw = pd.read_csv(download_csv(name))
    smiles_col = resolve_column(raw.columns, DATASET_SPECS[name]["smiles_columns"])
    target_col = resolve_column(raw.columns, DATASET_SPECS[name]["target_columns"])
    records = []
    for source_index, row in raw.iterrows():
        canonical, mol = canonicalize_smiles(row.get(smiles_col))
        target = pd.to_numeric(pd.Series([row.get(target_col)]), errors="coerce").iloc[0]
        if canonical is None or mol is None or not np.isfinite(target):
            continue
        records.append({
            "source_index": int(source_index),
            "smiles": canonical,
            "target": float(target),
            "scaffold": molecular_group(mol, canonical),
        })
    raw_frame = pd.DataFrame(records)
    kept = []
    for smiles, group in raw_frame.groupby("smiles", sort=False):
        targets = group["target"].to_numpy(dtype=float)
        if DATASET_SPECS[name]["task"] == "classification":
            conflict = len(np.unique(targets.astype(int))) > 1
        else:
            conflict = np.nanmax(targets) - np.nanmin(targets) > 1e-8
        if not conflict:
            kept.append(group.iloc[0])
    frame = pd.DataFrame(kept).reset_index(drop=True)
    frame["stable_index"] = np.arange(len(frame), dtype=int)
    frame["sample_id"] = [f"{name}_{hashlib.sha1(s.encode()).hexdigest()[:14]}" for s in frame["smiles"]]
    if DATASET_SPECS[name]["task"] == "classification":
        frame["target"] = frame["target"].astype(int)
    frame.to_csv(cache_path, index=False)
    return frame

frames = {name: load_frame(name) for name in DATASET_SPECS}
for name, frame in frames.items():
    print(name, len(frame), frame_fingerprint(frame)[:16], frame["target"].mean())
    if not prior_predictions.empty:
        expected = set(prior_predictions[prior_predictions["dataset"] == name]["sample_id"].astype(str))
        actual = set(frame["sample_id"].astype(str))
        if expected and not expected.issubset(actual):
            raise RuntimeError(f"{name} sample IDs do not align with the completed experiment.")

stamp_end("Data", _t0)


[START] 2026-09-20 23:12:11 | Data


[23:12:12] Explicit valence for atom # 1 N, 4, is greater than permitted
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] Explicit valence for atom # 6 N, 4, is greater than permitted
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] Explicit valence for atom # 6 N, 4, is greater than permitted
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] WARNING: not removing hydrogen atom without neighbors
[23:12:12] Explicit valence for atom # 11 N, 4, is greater than pe

BBBP 1965 b3359fe89e492fdd 0.7633587786259542
BACE 1513 0d9adb06889ff9cc 0.456708526107072
ESOL 1111 315d80a864ada969 -3.056288928892889
[DONE] 2026-09-20 23:12:17 | Data | 6.19s


## Splits


In [8]:
_t0 = stamp_start("Splits")

def stratification_labels(targets, task, max_bins=10):
    if task == "classification":
        return targets.astype(int)
    unique = len(np.unique(targets))
    bins = min(max_bins, max(2, unique // 10))
    ranked = pd.Series(targets).rank(method="first")
    return pd.qcut(ranked, q=bins, labels=False, duplicates="drop").to_numpy()

def make_outer_folds(name):
    frame = frames[name]
    task = DATASET_SPECS[name]["task"]
    y_split = stratification_labels(frame["target"].to_numpy(), task)
    groups = frame["scaffold"].to_numpy()
    n = len(frame)
    expected_fraction = 1.0 / CFG.folds
    candidates = []
    for offset in range(50):
        splitter = StratifiedGroupKFold(n_splits=CFG.folds, shuffle=True, random_state=CFG.split_seed + offset)
        folds = [(a.astype(int), b.astype(int)) for a, b in splitter.split(frame, y_split, groups)]
        valid = True
        size_penalty = 0.0
        target_penalty = 0.0
        overall = float(frame["target"].mean())
        for train, test in folds:
            size_penalty += abs(len(test) / n - expected_fraction)
            if task == "classification":
                if len(np.unique(frame.iloc[train]["target"])) < 2 or len(np.unique(frame.iloc[test]["target"])) < 2:
                    valid = False
                    break
                target_penalty += abs(float(frame.iloc[test]["target"].mean()) - overall)
        if valid:
            candidates.append((size_penalty + target_penalty, folds))
    if not candidates:
        raise RuntimeError(name)
    return min(candidates, key=lambda item: item[0])[1]

def make_inner_split(name, outer_train, fold):
    frame = frames[name]
    task = DATASET_SPECS[name]["task"]
    subset = frame.iloc[outer_train]
    y_split = stratification_labels(subset["target"].to_numpy(), task)
    groups = subset["scaffold"].to_numpy()
    n_splits = min(5, int(subset["scaffold"].nunique()))
    candidates = []
    desired = 0.20
    overall = float(subset["target"].mean())
    for offset in range(30):
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG.split_seed + 100 * fold + offset)
        for train_local, val_local in splitter.split(subset, y_split, groups):
            if task == "classification":
                if len(np.unique(subset.iloc[train_local]["target"])) < 2 or len(np.unique(subset.iloc[val_local]["target"])) < 2:
                    continue
            penalty = abs(len(val_local) / len(subset) - desired) + abs(float(subset.iloc[val_local]["target"].mean()) - overall)
            candidates.append((penalty, train_local, val_local))
    if not candidates:
        raise RuntimeError((name, fold))
    _, train_local, val_local = min(candidates, key=lambda item: item[0])
    return outer_train[train_local], outer_train[val_local]

SPLITS = {}
for name in DATASET_SPECS:
    outer = make_outer_folds(name)
    for fold, (outer_train, test) in enumerate(outer):
        train, val = make_inner_split(name, outer_train, fold)
        SPLITS[(name, fold)] = {"train": train, "val": val, "test": test}
        print(name, fold, len(train), len(val), len(test))

stamp_end("Splits", _t0)


[START] 2026-09-20 23:12:17 | Splits
BBBP 0 1067 257 641
BBBP 1 1068 258 639
BBBP 2 1026 254 685
BACE 0 802 198 513
BACE 1 822 197 494
BACE 2 804 203 506
ESOL 0 539 134 438
ESOL 1 590 167 354
ESOL 2 632 160 319
[DONE] 2026-09-20 23:13:14 | Splits | 56.60s


## Features


In [9]:
_t0 = stamp_start("Features")

ATOM_NUMBERS = [1, 5, 6, 7, 8, 9, 15, 16, 17, 35, 53]
DEGREES = [0, 1, 2, 3, 4, 5]
FORMAL_CHARGES = [-2, -1, 0, 1, 2]
TOTAL_HS = [0, 1, 2, 3, 4]
HYBRIDIZATIONS = [
    Chem.rdchem.HybridizationType.SP,
    Chem.rdchem.HybridizationType.SP2,
    Chem.rdchem.HybridizationType.SP3,
    Chem.rdchem.HybridizationType.SP3D,
    Chem.rdchem.HybridizationType.SP3D2,
]
CHIRAL_TAGS = [
    Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
    Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
]


def one_hot_other(value, choices):
    return [float(value == c) for c in choices] + [float(value not in choices)]


def atom_features(atom):
    return (
        one_hot_other(atom.GetAtomicNum(), ATOM_NUMBERS)
        + one_hot_other(atom.GetTotalDegree(), DEGREES)
        + one_hot_other(atom.GetFormalCharge(), FORMAL_CHARGES)
        + one_hot_other(atom.GetTotalNumHs(), TOTAL_HS)
        + one_hot_other(atom.GetHybridization(), HYBRIDIZATIONS)
        + one_hot_other(atom.GetChiralTag(), CHIRAL_TAGS)
        + [float(atom.GetIsAromatic()), float(atom.IsInRing())]
    )


ATOM_DIM = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))


def bond_value(bond):
    return {
        Chem.rdchem.BondType.SINGLE: 1.0,
        Chem.rdchem.BondType.DOUBLE: 1.35,
        Chem.rdchem.BondType.TRIPLE: 1.6,
        Chem.rdchem.BondType.AROMATIC: 1.2,
    }.get(bond.GetBondType(), 1.0)


def graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    x = np.asarray([atom_features(atom) for atom in mol.GetAtoms()], dtype=np.float32)
    n = len(x)
    adjacency = np.zeros((n, n), dtype=np.float32)
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        value = bond_value(bond)
        adjacency[i, j] = value
        adjacency[j, i] = value
    np.fill_diagonal(adjacency, 1.0)
    return x, adjacency


def fragment_hamiltonian(smiles):
    mol = Chem.MolFromSmiles(smiles)
    atoms = list(mol.GetAtoms())
    scores = []
    for atom in atoms:
        score = 10.0 * atom.GetTotalDegree() + 2.0 * float(atom.GetIsAromatic()) + 0.01 * atom.GetAtomicNum()
        scores.append((score, atom.GetIdx()))
    selected = [idx for _, idx in sorted(scores, reverse=True)[:CFG.n_qubits]]
    while len(selected) < CFG.n_qubits:
        selected.append(-1)
    z = np.zeros(CFG.n_qubits, dtype=np.float32)
    zz = np.zeros(len(PAIR_LIST), dtype=np.float32)
    hop = np.zeros(len(PAIR_LIST), dtype=np.float32)
    distance = Chem.GetDistanceMatrix(mol)
    for q, atom_idx in enumerate(selected):
        if atom_idx < 0:
            continue
        atom = mol.GetAtomWithIdx(atom_idx)
        value = 0.12 * (atom.GetAtomicNum() - 6) + 0.18 * atom.GetFormalCharge() + 0.03 * (atom.GetTotalDegree() - 2) + 0.02 * float(atom.GetIsAromatic())
        z[q] = np.clip(value, -1.5, 1.5)
    for k, (q1, q2) in enumerate(PAIR_LIST):
        a = selected[q1]
        b = selected[q2]
        if a < 0 or b < 0:
            continue
        bond = mol.GetBondBetweenAtoms(int(a), int(b))
        if bond is not None:
            value = bond_value(bond)
            hop[k] = 0.12 * value
            zz[k] = 0.04 * value
        else:
            d = float(distance[a, b])
            if np.isfinite(d) and d > 0:
                zz[k] = 0.015 / d
    norm = max(float(np.abs(z).max()), float(np.abs(zz).max()), float(np.abs(hop).max()), 1.0)
    return z / norm, zz / norm, hop / norm


GRAPH_CACHE = {}
for name, frame in frames.items():
    path = CACHE / f"{name}_graphs_qgnn_vqe.pt"
    if path.exists():
        GRAPH_CACHE[name] = torch.load(path, map_location="cpu", weights_only=False)
    else:
        records = []
        for row in frame.itertuples():
            x, adjacency = graph_from_smiles(row.smiles)
            hz, hzz, hhop = fragment_hamiltonian(row.smiles)
            records.append({
                "x": torch.tensor(x, dtype=torch.float32),
                "adj": torch.tensor(adjacency, dtype=torch.float32),
                "hz": torch.tensor(hz, dtype=torch.float32),
                "hzz": torch.tensor(hzz, dtype=torch.float32),
                "hhop": torch.tensor(hhop, dtype=torch.float32),
                "y": float(row.target),
                "sample_id": str(row.sample_id),
                "scaffold": str(row.scaffold),
            })
        torch.save(records, path)
        GRAPH_CACHE[name] = records
    print(name, len(GRAPH_CACHE[name]), ATOM_DIM)


class GraphDataset(Dataset):
    def __init__(self, records, indices):
        self.records = records
        self.indices = np.asarray(indices, dtype=int)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        return self.records[int(self.indices[i])]


def collate_graph(batch):
    b = len(batch)
    max_n = max(item["x"].shape[0] for item in batch)
    x = torch.zeros(b, max_n, ATOM_DIM, dtype=torch.float32)
    adj = torch.zeros(b, max_n, max_n, dtype=torch.float32)
    mask = torch.zeros(b, max_n, dtype=torch.bool)
    y = torch.tensor([item["y"] for item in batch], dtype=torch.float32)
    hz = torch.stack([item["hz"] for item in batch])
    hzz = torch.stack([item["hzz"] for item in batch])
    hhop = torch.stack([item["hhop"] for item in batch])
    sample_ids = []
    scaffolds = []
    for i, item in enumerate(batch):
        n = item["x"].shape[0]
        x[i, :n] = item["x"]
        adj[i, :n, :n] = item["adj"]
        mask[i, :n] = True
        sample_ids.append(item["sample_id"])
        scaffolds.append(item["scaffold"])
    return {"x": x, "adj": adj, "mask": mask, "hz": hz, "hzz": hzz, "hhop": hhop, "y": y, "sample_id": sample_ids, "scaffold": scaffolds}


def make_loaders(name, fold, seed):
    split = SPLITS[(name, fold)]
    records = GRAPH_CACHE[name]
    generator = torch.Generator().manual_seed(seed)
    return {
        "train": DataLoader(GraphDataset(records, split["train"]), batch_size=CFG.batch_size, shuffle=True, generator=generator, collate_fn=collate_graph, num_workers=0),
        "train_eval": DataLoader(GraphDataset(records, split["train"]), batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_graph, num_workers=0),
        "val": DataLoader(GraphDataset(records, split["val"]), batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_graph, num_workers=0),
        "test": DataLoader(GraphDataset(records, split["test"]), batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_graph, num_workers=0),
    }

stamp_end("Features", _t0)


[START] 2026-09-20 23:13:14 | Features


[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:14] WARNING: not removing hydrogen atom without neighbors
[23:13:15] WARNING: not removing hydrogen atom without neighbors
[23:13:15] WARNING: not r

BBBP 1965 43
BACE 1513 43
ESOL 1111 43
[DONE] 2026-09-20 23:13:21 | Features | 7.41s


## Hybrid QGNN–VQE


In [10]:
_t0 = stamp_start("Hybrid QGNN–VQE")

class GraphAttentionBlock(nn.Module):
    def __init__(self, hidden, heads, dropout):
        super().__init__()
        self.hidden = hidden
        self.heads = heads
        self.head_dim = hidden // heads
        self.qkv = nn.Linear(hidden, hidden * 3)
        self.out = nn.Linear(hidden, hidden)
        self.edge_scale = nn.Parameter(torch.ones(heads))
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, hidden * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, adj, mask):
        b, n, d = x.shape
        qkv = self.qkv(x).view(b, n, 3, self.heads, self.head_dim)
        q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        scores = scores + self.edge_scale.view(1, -1, 1, 1) * adj.unsqueeze(1)
        valid = (adj > 0).unsqueeze(1) & mask[:, None, :, None] & mask[:, None, None, :]
        scores = scores.masked_fill(~valid, -1e4)
        attention = torch.softmax(scores, dim=-1)
        out = torch.matmul(attention, v).permute(0, 2, 1, 3).reshape(b, n, d)
        x = self.norm1(x + self.dropout(self.out(out)))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x * mask.unsqueeze(-1)


class QGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input = nn.Linear(ATOM_DIM, CFG.hidden_dim)
        self.layers = nn.ModuleList([
            GraphAttentionBlock(CFG.hidden_dim, CFG.attention_heads, CFG.dropout)
            for _ in range(CFG.graph_layers)
        ])
        self.pool = nn.Sequential(
            nn.Linear(CFG.hidden_dim * 2, CFG.hidden_dim),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.bottleneck = nn.Sequential(
            nn.Linear(CFG.hidden_dim, CFG.bottleneck_dim),
            nn.LayerNorm(CFG.bottleneck_dim),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(CFG.bottleneck_dim, CFG.bottleneck_dim),
            nn.GELU(),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.bottleneck_dim, 1),
        )

    def forward(self, batch):
        x, adj, mask = batch["x"], batch["adj"], batch["mask"]
        h = self.input(x) * mask.unsqueeze(-1)
        for layer in self.layers:
            h = layer(h, adj, mask)
        denominator = mask.sum(1, keepdim=True).clamp_min(1).to(h.dtype)
        mean_pool = (h * mask.unsqueeze(-1)).sum(1) / denominator
        max_pool = h.masked_fill(~mask.unsqueeze(-1), -1e4).max(1).values
        graph = self.pool(torch.cat([mean_pool, max_pool], dim=-1))
        embedding = self.bottleneck(graph)
        prediction = self.head(embedding).squeeze(-1)
        return prediction, embedding


def basis_bit(index, qubit):
    return (index >> (CFG.n_qubits - 1 - qubit)) & 1


def flipped(index, qubits):
    value = index
    for q in qubits:
        value ^= 1 << (CFG.n_qubits - 1 - q)
    return value


def excitation_pairs_single(i, a):
    src, dst = [], []
    for index in range(2 ** CFG.n_qubits):
        if basis_bit(index, i) == 1 and basis_bit(index, a) == 0:
            src.append(index)
            dst.append(flipped(index, (i, a)))
    return torch.tensor(src, dtype=torch.long), torch.tensor(dst, dtype=torch.long)


def excitation_pairs_double(i, j, a, b):
    src, dst = [], []
    for index in range(2 ** CFG.n_qubits):
        if basis_bit(index, i) == 1 and basis_bit(index, j) == 1 and basis_bit(index, a) == 0 and basis_bit(index, b) == 0:
            src.append(index)
            dst.append(flipped(index, (i, j, a, b)))
    return torch.tensor(src, dtype=torch.long), torch.tensor(dst, dtype=torch.long)


EXCITATION_PAIRS = []
for item in SINGLES:
    EXCITATION_PAIRS.append(excitation_pairs_single(*item))
for item in DOUBLES:
    EXCITATION_PAIRS.append(excitation_pairs_double(*item))
EXCITATION_PAIRS = [(a.to(DEVICE), b.to(DEVICE)) for a, b in EXCITATION_PAIRS]

indices = torch.arange(2 ** CFG.n_qubits, device=DEVICE)
z_signs = []
for q in range(CFG.n_qubits):
    bit = (indices >> (CFG.n_qubits - 1 - q)) & 1
    z_signs.append((1 - 2 * bit).float())
Z_SIGNS = torch.stack(z_signs, dim=0)
ZZ_SIGNS = torch.stack([Z_SIGNS[i] * Z_SIGNS[j] for i, j in PAIR_LIST], dim=0)
HOPPING_PAIRS = []
for i, j in PAIR_LIST:
    src, dst = [], []
    for index in range(2 ** CFG.n_qubits):
        if basis_bit(index, i) == 0 and basis_bit(index, j) == 1:
            src.append(index)
            dst.append(flipped(index, (i, j)))
    HOPPING_PAIRS.append((torch.tensor(src, dtype=torch.long, device=DEVICE), torch.tensor(dst, dtype=torch.long, device=DEVICE)))


def apply_excitation(state, theta, src, dst):
    angle = theta / CFG.vqe_depth
    c = torch.cos(angle / 2).unsqueeze(-1)
    s = torch.sin(angle / 2).unsqueeze(-1)
    left = state[:, src]
    right = state[:, dst]
    out = state.clone()
    out[:, src] = c * left - s * right
    out[:, dst] = s * left + c * right
    return out


def uccsd_state(theta):
    b = theta.shape[0]
    state = torch.zeros(b, 2 ** CFG.n_qubits, dtype=torch.complex64, device=theta.device)
    hf_index = sum(1 << (CFG.n_qubits - 1 - q) for q in OCCUPIED)
    state[:, hf_index] = 1.0 + 0.0j
    for _ in range(CFG.vqe_depth):
        for p, (src, dst) in enumerate(EXCITATION_PAIRS):
            state = apply_excitation(state, theta[:, p], src, dst)
    return state


def quantum_expectations(state):
    probabilities = state.abs().pow(2)
    z = probabilities @ Z_SIGNS.t()
    zz = probabilities @ ZZ_SIGNS.t()
    hops = []
    for src, dst in HOPPING_PAIRS:
        value = 4.0 * torch.real((state[:, src].conj() * state[:, dst]).sum(-1))
        hops.append(value)
    hop = torch.stack(hops, dim=-1)
    return z, zz, hop


def hamiltonian_energy(state, hz, hzz, hhop):
    z, zz, hop = quantum_expectations(state)
    return (z * hz).sum(-1) + (zz * hzz).sum(-1) + (hop * hhop).sum(-1)


def shot_sample(expectation, scale=1.0):
    normalized = (expectation / scale).clamp(-1.0, 1.0)
    probability = ((normalized + 1.0) / 2.0).clamp(1e-6, 1 - 1e-6)
    counts = torch.binomial(torch.full_like(probability, float(CFG.shots)), probability)
    return scale * (2.0 * counts / float(CFG.shots) - 1.0)


def optimize_vqe(initial_theta, hz, hzz, hhop):
    theta = initial_theta.detach().clone().to(DEVICE).requires_grad_(True)
    optimizer = torch.optim.Adam([theta], lr=CFG.vqe_learning_rate)
    for _ in range(CFG.vqe_iterations):
        optimizer.zero_grad(set_to_none=True)
        state = uccsd_state(theta)
        energy = hamiltonian_energy(state, hz, hzz, hhop)
        energy.sum().backward()
        optimizer.step()
    with torch.no_grad():
        state = uccsd_state(theta)
        z, zz, hop = quantum_expectations(state)
        z_s = shot_sample(z, 1.0)
        zz_s = shot_sample(zz, 1.0)
        hop_s = shot_sample(hop, 2.0)
        energy_s = (z_s * hz).sum(-1, keepdim=True) + (zz_s * hzz).sum(-1, keepdim=True) + (hop_s * hhop).sum(-1, keepdim=True)
        return torch.cat([z_s, zz_s, hop_s, energy_s], dim=-1).cpu()


class FusionHead(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


print("Graph layers:", CFG.graph_layers)
print("Attention heads:", CFG.attention_heads)
print("Hidden dimension:", CFG.hidden_dim)
print("Bottleneck:", CFG.bottleneck_dim)
print("Qubits:", CFG.n_qubits)
print("UCCSD Trotter depth:", CFG.vqe_depth)
print("VQE iterations:", CFG.vqe_iterations)
print("VQE learning rate:", CFG.vqe_learning_rate)
print("Shots:", CFG.shots)
print("PCA components:", CFG.pca_components)
stamp_end("Hybrid QGNN–VQE", _t0)


[START] 2026-09-20 23:13:21 | Hybrid QGNN–VQE
Graph layers: 6
Attention heads: 8
Hidden dimension: 256
Bottleneck: 64
Qubits: 6
UCCSD Trotter depth: 8
VQE iterations: 200
VQE learning rate: 0.01
Shots: 1024
PCA components: 10
[DONE] 2026-09-20 23:13:22 | Hybrid QGNN–VQE | 0.46s


## Training


In [11]:
_t0 = stamp_start("Training")


def task_state(name, fold):
    train_idx = SPLITS[(name, fold)]["train"]
    y = frames[name].iloc[train_idx]["target"].to_numpy(dtype=float)
    if DATASET_SPECS[name]["task"] == "classification":
        positives = y.sum()
        negatives = len(y) - positives
        return {"mean": 0.0, "std": 1.0, "pos_weight": float(negatives / max(positives, 1))}
    return {"mean": float(y.mean()), "std": float(max(y.std(), 1e-8)), "pos_weight": 1.0}


def move_batch(batch):
    return {
        "x": batch["x"].to(DEVICE),
        "adj": batch["adj"].to(DEVICE),
        "mask": batch["mask"].to(DEVICE),
        "hz": batch["hz"].to(DEVICE),
        "hzz": batch["hzz"].to(DEVICE),
        "hhop": batch["hhop"].to(DEVICE),
        "y": batch["y"].to(DEVICE),
        "sample_id": batch["sample_id"],
        "scaffold": batch["scaffold"],
    }


def loss_value(pred, y, task, state):
    if task == "classification":
        pos_weight = torch.tensor(state["pos_weight"], dtype=pred.dtype, device=pred.device)
        return F.binary_cross_entropy_with_logits(pred, y, pos_weight=pos_weight)
    target = (y - state["mean"]) / state["std"]
    return F.smooth_l1_loss(pred, target, beta=0.6)


def metric_value(y, p, task):
    if task == "classification":
        return float(roc_auc_score(y, p))
    return float(mean_squared_error(y, p) ** 0.5)


def better(score, best, task):
    if best is None:
        return True
    return score > best + 1e-5 if task == "classification" else score < best - 1e-5


@torch.no_grad()
def qgnn_predict(model, loader, task, state):
    model.eval()
    ys, preds = [], []
    for raw in loader:
        batch = move_batch(raw)
        output, _ = model(batch)
        value = torch.sigmoid(output) if task == "classification" else output * state["std"] + state["mean"]
        ys.extend(batch["y"].cpu().numpy().tolist())
        preds.extend(value.cpu().numpy().tolist())
    return np.asarray(ys), np.asarray(preds)


def train_qgnn(name, fold, seed, loaders, state, directory):
    task = DATASET_SPECS[name]["task"]
    model = QGNN().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.graph_learning_rate, weight_decay=CFG.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.graph_epochs)
    best = None
    bad = 0
    best_path = directory / "qgnn_best.pt"
    for epoch in range(1, CFG.graph_epochs + 1):
        model.train()
        losses = []
        for raw in loaders["train"]:
            batch = move_batch(raw)
            optimizer.zero_grad(set_to_none=True)
            pred, _ = model(batch)
            loss = loss_value(pred, batch["y"], task, state)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        scheduler.step()
        val_y, val_p = qgnn_predict(model, loaders["val"], task, state)
        score = metric_value(val_y, val_p, task)
        improved = better(score, best, task)
        if improved:
            best = score
            bad = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "score": score}, best_path)
        else:
            bad += 1
        if epoch == 1 or epoch % 25 == 0 or improved:
            print(f"{name} | fold {fold} | seed {seed} | QGNN epoch {epoch} | train {np.mean(losses):.5f} | val {score:.5f} | best {best:.5f}", flush=True)
        if bad >= CFG.graph_patience:
            break
    payload = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(payload["model"])
    return model, int(payload["epoch"]), float(payload["score"])


@torch.no_grad()
def extract_qgnn(model, loader, task, state):
    model.eval()
    ys, raw_predictions, embeddings, hz, hzz, hhop, sample_ids, scaffolds = [], [], [], [], [], [], [], []
    for raw in loader:
        batch = move_batch(raw)
        output, embedding = model(batch)
        prediction = torch.sigmoid(output) if task == "classification" else output * state["std"] + state["mean"]
        ys.append(batch["y"].cpu())
        raw_predictions.append(prediction.cpu())
        embeddings.append(embedding.cpu())
        hz.append(batch["hz"].cpu())
        hzz.append(batch["hzz"].cpu())
        hhop.append(batch["hhop"].cpu())
        sample_ids.extend(raw["sample_id"])
        scaffolds.extend(raw["scaffold"])
    return {
        "y": torch.cat(ys).numpy(),
        "prediction": torch.cat(raw_predictions).numpy(),
        "embedding": torch.cat(embeddings).numpy(),
        "hz": torch.cat(hz).numpy(),
        "hzz": torch.cat(hzz).numpy(),
        "hhop": torch.cat(hhop).numpy(),
        "sample_id": sample_ids,
        "scaffold": scaffolds,
    }


def pca_initialization(train_embedding, split_embeddings):
    components = min(N_VQE_PARAMS, train_embedding.shape[1], max(1, len(train_embedding) - 1))
    mapper = PCA(n_components=components, random_state=0)
    mapper.fit(train_embedding)
    train_scale = mapper.transform(train_embedding).std(axis=0, keepdims=True)
    train_scale[train_scale < 1e-8] = 1.0
    outputs = []
    for embedding in split_embeddings:
        projected = mapper.transform(embedding) / train_scale
        if projected.shape[1] < N_VQE_PARAMS:
            projected = np.pad(projected, ((0, 0), (0, N_VQE_PARAMS - projected.shape[1])))
        outputs.append((0.20 * np.pi * np.tanh(projected[:, :N_VQE_PARAMS])).astype(np.float32))
    return outputs


def similarity_features(train_embedding, split_embeddings):
    components = min(CFG.pca_components, train_embedding.shape[1], max(1, len(train_embedding) - 1))
    mapper = PCA(n_components=components, random_state=0)
    train_pca = mapper.fit_transform(train_embedding)
    results = []
    for index, embedding in enumerate(split_embeddings):
        transformed = mapper.transform(embedding)
        sim = cosine_similarity(transformed, train_pca)
        if index == 0 and len(sim) == len(train_pca):
            np.fill_diagonal(sim, -1.0)
        results.append(sim.max(axis=1, keepdims=True).astype(np.float32))
    return results


def quantum_features(initial, hz, hzz, hhop, label):
    output = []
    total = len(initial)
    for start in range(0, total, CFG.batch_size):
        stop = min(start + CFG.batch_size, total)
        theta = torch.tensor(initial[start:stop], dtype=torch.float32, device=DEVICE)
        hz_b = torch.tensor(hz[start:stop], dtype=torch.float32, device=DEVICE)
        hzz_b = torch.tensor(hzz[start:stop], dtype=torch.float32, device=DEVICE)
        hhop_b = torch.tensor(hhop[start:stop], dtype=torch.float32, device=DEVICE)
        output.append(optimize_vqe(theta, hz_b, hzz_b, hhop_b))
        print(f"{label} | VQE {stop}/{total}", flush=True)
    return torch.cat(output).numpy().astype(np.float32)


def build_fusion_features(parts, q_features, similarity):
    return np.concatenate([
        parts["embedding"].astype(np.float32),
        parts["prediction"].reshape(-1, 1).astype(np.float32),
        q_features.astype(np.float32),
        similarity.astype(np.float32),
    ], axis=1)


def train_fusion(name, fold, seed, train_x, train_y, val_x, val_y, state, directory):
    task = DATASET_SPECS[name]["task"]
    model = FusionHead(train_x.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.fusion_learning_rate, weight_decay=CFG.weight_decay)
    train_x_t = torch.tensor(train_x, dtype=torch.float32)
    train_y_t = torch.tensor(train_y, dtype=torch.float32)
    val_x_t = torch.tensor(val_x, dtype=torch.float32, device=DEVICE)
    batch = CFG.batch_size
    best = None
    bad = 0
    best_path = directory / "fusion_best.pt"
    generator = torch.Generator().manual_seed(seed)
    for epoch in range(1, CFG.fusion_epochs + 1):
        model.train()
        order = torch.randperm(len(train_x_t), generator=generator)
        losses = []
        for start in range(0, len(order), batch):
            idx = order[start:start + batch]
            xb = train_x_t[idx].to(DEVICE)
            yb = train_y_t[idx].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_value(pred, yb, task, state)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        model.eval()
        with torch.no_grad():
            raw = model(val_x_t)
            pred = torch.sigmoid(raw).cpu().numpy() if task == "classification" else (raw * state["std"] + state["mean"]).cpu().numpy()
        score = metric_value(val_y, pred, task)
        improved = better(score, best, task)
        if improved:
            best = score
            bad = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "score": score, "input_dim": train_x.shape[1]}, best_path)
        else:
            bad += 1
        if epoch == 1 or epoch % 20 == 0 or improved:
            print(f"{name} | fold {fold} | seed {seed} | fusion epoch {epoch} | train {np.mean(losses):.5f} | val {score:.5f} | best {best:.5f}", flush=True)
        if bad >= CFG.fusion_patience:
            break
    payload = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(payload["model"])
    return model, int(payload["epoch"]), float(payload["score"])


def train_hybrid_qgnn_vqe(name, fold, seed):
    seed_all(seed)
    directory = CHECKPOINTS / name / f"fold{fold}_seed{seed}"
    directory.mkdir(parents=True, exist_ok=True)
    task = DATASET_SPECS[name]["task"]
    state = task_state(name, fold)
    loaders = make_loaders(name, fold, seed)
    qgnn, qgnn_epoch, qgnn_val = train_qgnn(name, fold, seed, loaders, state, directory)
    train_part = extract_qgnn(qgnn, loaders["train_eval"], task, state)
    val_part = extract_qgnn(qgnn, loaders["val"], task, state)
    test_part = extract_qgnn(qgnn, loaders["test"], task, state)
    embeddings = [train_part["embedding"], val_part["embedding"], test_part["embedding"]]
    initializations = pca_initialization(train_part["embedding"], embeddings)
    similarities = similarity_features(train_part["embedding"], embeddings)
    quantum_cache = directory / "vqe_features.npz"
    if quantum_cache.exists():
        cached = np.load(quantum_cache)
        train_q = cached["train"]
        val_q = cached["val"]
        test_q = cached["test"]
        print(f"[VQE CACHE] {name} fold={fold} seed={seed}", flush=True)
    else:
        train_q = quantum_features(initializations[0], train_part["hz"], train_part["hzz"], train_part["hhop"], f"{name} fold {fold} seed {seed} train")
        val_q = quantum_features(initializations[1], val_part["hz"], val_part["hzz"], val_part["hhop"], f"{name} fold {fold} seed {seed} val")
        test_q = quantum_features(initializations[2], test_part["hz"], test_part["hzz"], test_part["hhop"], f"{name} fold {fold} seed {seed} test")
        np.savez_compressed(quantum_cache, train=train_q, val=val_q, test=test_q)
    train_x = build_fusion_features(train_part, train_q, similarities[0])
    val_x = build_fusion_features(val_part, val_q, similarities[1])
    test_x = build_fusion_features(test_part, test_q, similarities[2])
    fusion, fusion_epoch, fusion_val = train_fusion(name, fold, seed, train_x, train_part["y"], val_x, val_part["y"], state, directory)
    fusion.eval()
    with torch.no_grad():
        raw = fusion(torch.tensor(test_x, dtype=torch.float32, device=DEVICE))
        prediction = torch.sigmoid(raw).cpu().numpy() if task == "classification" else (raw * state["std"] + state["mean"]).cpu().numpy()
    y = test_part["y"]
    result = {
        "dataset": name,
        "fold": fold,
        "seed": seed,
        "model": BASE_NAME,
        "task": task,
        "qgnn_best_epoch": qgnn_epoch,
        "qgnn_validation_primary": qgnn_val,
        "fusion_best_epoch": fusion_epoch,
        "validation_primary": fusion_val,
    }
    if task == "classification":
        result.update({
            "ROC_AUC": float(roc_auc_score(y, prediction)),
            "PR_AUC": float(average_precision_score(y, prediction)),
            "Accuracy": float(accuracy_score(y, (prediction >= 0.5).astype(int))),
            "F1": float(f1_score(y, (prediction >= 0.5).astype(int))),
        })
    else:
        result.update({
            "RMSE": float(mean_squared_error(y, prediction) ** 0.5),
            "MAE": float(mean_absolute_error(y, prediction)),
            "R2": float(r2_score(y, prediction)),
        })
    predictions = pd.DataFrame({
        "dataset": name,
        "fold": fold,
        "seed": seed,
        "model": BASE_NAME,
        "sample_id": test_part["sample_id"],
        "scaffold": test_part["scaffold"],
        "target": y,
        "prediction": prediction,
    })
    return result, predictions

stamp_end("Training", _t0)


[START] 2026-09-20 23:13:22 | Training
[DONE] 2026-09-20 23:13:22 | Training | 0.01s


## Run


In [ ]:
_t0 = stamp_start("Run")

metric_file = RESULTS / "hybrid_qgnn_vqe_metrics.csv"
prediction_file = RESULTS / "hybrid_qgnn_vqe_predictions.csv"

base_metrics = pd.read_csv(metric_file) if metric_file.exists() else pd.DataFrame()
base_predictions = pd.read_csv(prediction_file) if prediction_file.exists() else pd.DataFrame()
completed = set()
if not base_metrics.empty:
    completed = set(zip(base_metrics["dataset"], base_metrics["fold"].astype(int), base_metrics["seed"].astype(int)))

for name in DATASET_SPECS:
    for fold in range(CFG.folds):
        for seed in CFG.seeds:
            key = (name, fold, seed)
            if key in completed:
                print(f"[SKIP] {name} fold={fold} seed={seed}", flush=True)
                continue
            job_start = time.perf_counter()
            print(f"[JOB START] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {name} fold={fold} seed={seed}", flush=True)
            result, predictions = train_hybrid_qgnn_vqe(name, fold, seed)
            result["training_seconds"] = time.perf_counter() - job_start
            base_metrics = pd.concat([base_metrics, pd.DataFrame([result])], ignore_index=True)
            base_predictions = pd.concat([base_predictions, predictions], ignore_index=True)
            base_metrics.to_csv(metric_file, index=False)
            base_predictions.to_csv(prediction_file, index=False)
            print(f"[JOB DONE] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {name} fold={fold} seed={seed} | {result}", flush=True)

stamp_end("Run", _t0)


[START] 2026-09-20 23:13:22 | Run
[JOB START] 2026-09-20 23:13:22 | BBBP fold=0 seed=19
BBBP | fold 0 | seed 19 | QGNN epoch 1 | train 0.33148 | val 0.58556 | best 0.58556
BBBP | fold 0 | seed 19 | QGNN epoch 2 | train 0.32889 | val 0.75820 | best 0.75820
BBBP | fold 0 | seed 19 | QGNN epoch 3 | train 0.32683 | val 0.76422 | best 0.76422
BBBP | fold 0 | seed 19 | QGNN epoch 4 | train 0.32443 | val 0.77091 | best 0.77091
BBBP | fold 0 | seed 19 | QGNN epoch 5 | train 0.32738 | val 0.77877 | best 0.77877
BBBP | fold 0 | seed 19 | QGNN epoch 6 | train 0.32584 | val 0.77890 | best 0.77890
BBBP | fold 0 | seed 19 | QGNN epoch 25 | train 0.31437 | val 0.51393 | best 0.77890
BBBP fold 0 seed 19 train | VQE 48/1067
BBBP fold 0 seed 19 train | VQE 96/1067
BBBP fold 0 seed 19 train | VQE 144/1067
BBBP fold 0 seed 19 train | VQE 192/1067
BBBP fold 0 seed 19 train | VQE 240/1067
BBBP fold 0 seed 19 train | VQE 288/1067
BBBP fold 0 seed 19 train | VQE 336/1067
BBBP fold 0 seed 19 train | VQE 384/10

## Results


In [18]:
_t0 = stamp_start("Results")

prior_selected = prior_metrics[prior_metrics["model"].isin(["MLP", "GNN", "Proposed Topology-Aware Quantum"])].copy()
common_columns = sorted(set(prior_selected.columns).union(base_metrics.columns))
for column in common_columns:
    if column not in prior_selected:
        prior_selected[column] = np.nan
    if column not in base_metrics:
        base_metrics[column] = np.nan

all_metrics = pd.concat([prior_selected[common_columns], base_metrics[common_columns]], ignore_index=True)


def primary_metric(name):
    return "ROC_AUC" if DATASET_SPECS[name]["task"] == "classification" else "RMSE"


rows = []
for (dataset, model), group in all_metrics.groupby(["dataset", "model"]):
    metric = primary_metric(dataset)
    values = pd.to_numeric(group[metric], errors="coerce").dropna().to_numpy()
    if len(values) == 0:
        continue
    rows.append({
        "dataset": dataset,
        "model": model,
        "metric": metric,
        "jobs": len(values),
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)) if len(values) > 1 else np.nan,
    })

summary = pd.DataFrame(rows)
order = ["MLP", "GNN", BASE_NAME, "Proposed Topology-Aware Quantum"]
summary["model"] = pd.Categorical(summary["model"], categories=order, ordered=True)
summary = summary.sort_values(["dataset", "model"]).reset_index(drop=True)
summary.to_csv(RESULTS / "final_four_model_summary.csv", index=False)
display(summary)

stamp_end("Results", _t0)


[START] 2026-09-21 07:59:07 | Results


,dataset,model,metric,jobs,mean,std
0,BACE,MLP,ROC_AUC,6,0.840156,0.023356
1,BACE,GNN,ROC_AUC,6,0.766515,0.054804
2,BACE,Hybrid QGNN–VQE,ROC_AUC,6,0.681716,0.039356
3,BACE,Proposed Topology-Aware Quantum,ROC_AUC,6,0.859359,0.025811
4,BBBP,MLP,ROC_AUC,6,0.890419,0.018926
5,BBBP,GNN,ROC_AUC,6,0.869100,0.024477
6,BBBP,Hybrid QGNN–VQE,ROC_AUC,6,0.774001,0.067284
7,BBBP,Proposed Topology-Aware Quantum,ROC_AUC,6,0.906887,0.018021
8,ESOL,MLP,RMSE,6,1.300445,0.125604
9,ESOL,GNN,RMSE,6,0.849910,0.168010


[DONE] 2026-09-21 07:59:07 | Results | 0.02s


## Comparison


In [19]:
_t0 = stamp_start("Comparison")

comparison_rows = []
for dataset in DATASET_SPECS:
    subset = summary[summary["dataset"] == dataset]
    base_row = subset[subset["model"] == BASE_NAME]
    proposed_row = subset[subset["model"] == "Proposed Topology-Aware Quantum"]
    if base_row.empty or proposed_row.empty:
        continue
    metric = primary_metric(dataset)
    base_value = float(base_row["mean"].iloc[0])
    proposed_value = float(proposed_row["mean"].iloc[0])
    favorable = proposed_value - base_value if metric == "ROC_AUC" else base_value - proposed_value
    comparison_rows.append({
        "dataset": dataset,
        "metric": metric,
        "Hybrid QGNN–VQE": base_value,
        "Proposed": proposed_value,
        "favorable_difference_for_proposed": favorable,
        "proposed_better": bool(favorable > 0),
    })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULTS / "proposed_vs_hybrid_qgnn_vqe.csv", index=False)
display(comparison)

stamp_end("Comparison", _t0)


[START] 2026-09-21 07:59:13 | Comparison


,dataset,metric,Hybrid QGNN–VQE,Proposed,favorable_difference_for_proposed,proposed_better
0,BBBP,ROC_AUC,0.774001,0.906887,0.132887,True
1,BACE,ROC_AUC,0.681716,0.859359,0.177643,True
2,ESOL,RMSE,1.708610,0.737115,0.971495,True


[DONE] 2026-09-21 07:59:13 | Comparison | 0.01s


## Audit


In [20]:
_t0 = stamp_start("Audit")

implemented = {
    "graph_layers": CFG.graph_layers,
    "attention_heads": CFG.attention_heads,
    "hidden_dim": CFG.hidden_dim,
    "bottleneck_dim": CFG.bottleneck_dim,
    "n_qubits": CFG.n_qubits,
    "vqe_depth": CFG.vqe_depth,
    "vqe_iterations": CFG.vqe_iterations,
    "vqe_learning_rate": CFG.vqe_learning_rate,
    "shots": CFG.shots,
    "pca_components": CFG.pca_components,
}
print(json.dumps(implemented, indent=2))
print("Paper-aligned core: attention QGNN, graph pooling, 6-qubit UCCSD-style VQE, depth 8, 200 VQE iterations, Adam 0.01, 1024-shot final estimates, PCA-10 similarity.")
print("Task transfer: BBBP, BACE and ESOL replace the paper's QM9 IP/BFE targets and serine-ranking endpoint.")
print("Hamiltonian transfer: a deterministic six-qubit effective fragment Hamiltonian is constructed from molecular atom and bond information because ab-initio fragment integrals are not supplied for these MoleculeNet endpoints.")
print("Simulator gradient: exact autograd is used for VQE optimization; the paper describes parameter-shift gradients.")
print("Ranking-only QAOA and bijection stages are excluded because they do not define the BBBP, BACE or ESOL prediction tasks.")

stamp_end("Audit", _t0)


[START] 2026-09-21 08:04:14 | Audit
{
  "graph_layers": 6,
  "attention_heads": 8,
  "hidden_dim": 256,
  "bottleneck_dim": 64,
  "n_qubits": 6,
  "vqe_depth": 8,
  "vqe_iterations": 200,
  "vqe_learning_rate": 0.01,
  "shots": 1024,
  "pca_components": 10
}
Paper-aligned core: attention QGNN, graph pooling, 6-qubit UCCSD-style VQE, depth 8, 200 VQE iterations, Adam 0.01, 1024-shot final estimates, PCA-10 similarity.
Task transfer: BBBP, BACE and ESOL replace the paper's QM9 IP/BFE targets and serine-ranking endpoint.
Hamiltonian transfer: a deterministic six-qubit effective fragment Hamiltonian is constructed from molecular atom and bond information because ab-initio fragment integrals are not supplied for these MoleculeNet endpoints.
Simulator gradient: exact autograd is used for VQE optimization; the paper describes parameter-shift gradients.
Ranking-only QAOA and bijection stages are excluded because they do not define the BBBP, BACE or ESOL prediction tasks.
[DONE] 2026-09-21 08:0

## Export


In [21]:
_t0 = stamp_start("Export")

final_metrics = RESULTS / "final_four_model_metrics.csv"
all_metrics.to_csv(final_metrics, index=False)

config_file = RESULTS / "configuration.json"
config_file.write_text(json.dumps(asdict(CFG), indent=2))

resume_zip = Path("/kaggle/working/qgme_hybrid_qgnn_vqe_resume.zip")

if resume_zip.exists():
    resume_zip.unlink()

shutil.make_archive(
    str(resume_zip.with_suffix("")),
    "zip",
    root_dir=ROOT,
    base_dir="results"
)

print("Completed Hybrid QGNN-VQE jobs:", len(pd.read_csv(RESULTS / "hybrid_qgnn_vqe_metrics.csv")))
print("Metrics:", final_metrics)
print("Summary:", RESULTS / "final_four_model_summary.csv")
print("Comparison:", RESULTS / "proposed_vs_hybrid_qgnn_vqe.csv")
print("Resume backup:", resume_zip)

stamp_end("Export", _t0)

[START] 2026-09-21 08:05:44 | Export
Completed Hybrid QGNN-VQE jobs: 18
Metrics: /kaggle/working/qgme_hybrid_qgnn_vqe/results/final_four_model_metrics.csv
Summary: /kaggle/working/qgme_hybrid_qgnn_vqe/results/final_four_model_summary.csv
Comparison: /kaggle/working/qgme_hybrid_qgnn_vqe/results/proposed_vs_hybrid_qgnn_vqe.csv
Resume backup: /kaggle/working/qgme_hybrid_qgnn_vqe_resume.zip
[DONE] 2026-09-21 08:05:44 | Export | 0.04s


In [17]:
metric_file = RESULTS / "hybrid_qgnn_vqe_metrics.csv"
m = pd.read_csv(metric_file)
print("Completed jobs:", len(m))
print(m[["dataset", "fold", "seed", "model", "training_seconds"]].to_string(index=False))

Completed jobs: 18
dataset  fold  seed           model  training_seconds
   BBBP     0    19 Hybrid QGNN–VQE       1580.060444
   BBBP     0    53 Hybrid QGNN–VQE       1570.762702
   BBBP     1    19 Hybrid QGNN–VQE       1561.355868
   BBBP     1    53 Hybrid QGNN–VQE       1556.123371
   BBBP     2    19 Hybrid QGNN–VQE       1543.756521
   BBBP     2    53 Hybrid QGNN–VQE       1544.793565
   BACE     0    19 Hybrid QGNN–VQE       1184.890589
   BACE     0    53 Hybrid QGNN–VQE       1179.190648
   BACE     1    19 Hybrid QGNN–VQE       1231.066942
   BACE     1    53 Hybrid QGNN–VQE       1238.864566
   BACE     2    19 Hybrid QGNN–VQE       1193.113842
   BACE     2    53 Hybrid QGNN–VQE       1204.558181
   ESOL     0    19 Hybrid QGNN–VQE        886.852766
   ESOL     0    53 Hybrid QGNN–VQE        887.793433
   ESOL     1    19 Hybrid QGNN–VQE        889.420190
   ESOL     1    53 Hybrid QGNN–VQE        887.401742
   ESOL     2    19 Hybrid QGNN–VQE        883.913397
   ESOL  

In [22]:
import zipfile

_t0 = stamp_start("Full Backup")

full_zip = Path("/kaggle/working/qgme_hybrid_qgnn_vqe_full_backup.zip")

if full_zip.exists():
    full_zip.unlink()

folders = [RESULTS, CHECKPOINTS, FIGURES, CACHE]

with zipfile.ZipFile(
    full_zip,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:
    for folder in folders:
        if folder.exists():
            for path in folder.rglob("*"):
                if path.is_file():
                    zf.write(path, path.relative_to(ROOT))

with zipfile.ZipFile(full_zip, "r") as zf:
    names = zf.namelist()

print("Full backup:", full_zip)
print("Files:", len(names))
print("Checkpoint files:", len([x for x in names if x.endswith(".pt")]))
print("Results files:", len([x for x in names if x.startswith("results/")]))

stamp_end("Full Backup", _t0)

[START] 2026-09-21 08:14:03 | Full Backup
Full backup: /kaggle/working/qgme_hybrid_qgnn_vqe_full_backup.zip
Files: 66
Checkpoint files: 39
Results files: 6
[DONE] 2026-09-21 08:14:16 | Full Backup | 12.74s
